In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython import get_ipython
import tkinter as tk
from tkinter import filedialog
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
from scipy.interpolate import griddata
from plotly.subplots import make_subplots
import imageio
import imageio.v3 as imageio

import warnings
# Suppress any other unrelated deprecation warnings if needed
warnings.filterwarnings("ignore", category=DeprecationWarning)

try:
    # --- The Correct Fix ---
    # This is the ONLY import imageio statement needed.
    # It imports the v2 module as recommended by the warning.
    import imageio.v2 as imageio
    IMAGEIO_AVAILABLE = True
except ImportError:
    IMAGEIO_AVAILABLE = False
    print("⚠️ Warning: 'imageio' library not found. GIF export will be disabled.")
    print("   To enable GIF export, run '!pip install imageio' in a new notebook cell and restart the kernel.")


pio.renderers.default = "notebook"

# --- Helper Functions ---

def _parse_timestamp(series):
    """Attempt to parse a datetime series with multiple formats."""
    # Try specific format with fractional seconds first
    ts = pd.to_datetime(series, format="%d.%m.%Y %H:%M:%S.%f", errors="coerce")
    if ts.notna().any():
        return ts
    # Fallback to a more general, day-first parser
    return pd.to_datetime(series, dayfirst=True, errors="coerce")

def _generate_time_axis(num_points=1200, duration_seconds=30):
    """Helper to create a smooth time axis for test data."""
    time_deltas = pd.to_timedelta(np.linspace(0, duration_seconds, num_points), unit='s')
    start_time = pd.Timestamp.now().floor('s')
    return start_time + time_deltas

print("Libraries and helpers loaded. Plotly renderer and color scales configured.")

In [ ]:
def _create_base_test_df(config, test_name):
    """Creates the base DataFrame and extracts common variables for test data generation."""
    spatial_map = config["spatial_map"]
    all_columns = config["requested_columns"]
    control_col = config.get("control_thermocouple", "70418_T901000_W1T") # Use a default if not specified
    
    timestamps = _generate_time_axis()
    df = pd.DataFrame({'datetime': timestamps})
    
    # Get sensor positions and corresponding column names
    sensor_cols = [col for col in all_columns if col in spatial_map]
    sensor_positions = np.array([spatial_map[col] for col in sensor_cols])
    
    return df, timestamps, sensor_cols, sensor_positions, control_col, test_name

def generate_center_out_ripple_data(config):
    """VECTORIZED: Generates 'Fast Center-Out Ripple' data."""
    print("🔬 TEST MODE: Generating 'Fast Center-Out Ripple' data...")
    df, timestamps, sensor_cols, positions, control_col, test_name = _create_base_test_df(config, "Center-Out Ripple Test")

    base_temp, intensity = 800.0, 25.0
    center_pos = np.array([0.5, 0.5])
    
    # Vectorized calculation
    time_progress = np.linspace(0, 1, len(timestamps))[:, np.newaxis] # Shape: (n_times, 1)
    distances = np.linalg.norm(positions - center_pos, axis=1)      # Shape: (n_sensors,)
    
    # Broadcasting computes the effect for all times and all sensors at once
    effect = np.sin(distances * 30 - time_progress * 40) * intensity
    
    temp_data = base_temp + effect
    df[sensor_cols] = temp_data
    
    # Fill remaining columns and set control temperature
    for col in config["requested_columns"]:
        if col not in df.columns:
            df[col] = base_temp
    df[control_col] = base_temp
            
    df['source_file'] = test_name
    print("✅ Test data generated successfully.")
    return df

def generate_interfering_waves_data(config):
    """VECTORIZED: Generates 'Interfering Waves' data."""
    print("🔬 TEST MODE: Generating 'Interfering Waves' data...")
    df, timestamps, sensor_cols, positions, control_col, test_name = _create_base_test_df(config, "Interfering Waves Test")

    base_temp, intensity1, intensity2 = 800.0, 20.0, -20.0
    source1_pos, source2_pos = np.array([0.2, 0.75]), np.array([0.8, 0.25])

    # Vectorized calculation
    time_progress = np.linspace(0, 1, len(timestamps))[:, np.newaxis]
    dist1 = np.linalg.norm(positions - source1_pos, axis=1)
    dist2 = np.linalg.norm(positions - source2_pos, axis=1)

    effect1 = np.sin(dist1 * 25 - time_progress * 50) * intensity1
    effect2 = np.sin(dist2 * 35 - time_progress * 50) * intensity2
    
    temp_data = base_temp + effect1 + effect2
    df[sensor_cols] = temp_data

    for col in config["requested_columns"]:
        if col not in df.columns:
            df[col] = base_temp
    df[control_col] = base_temp
            
    df['source_file'] = test_name
    print("✅ Test data generated successfully.")
    return df

def generate_pulsing_hotspot_data(config):
    """VECTORIZED: Generates 'Pulsing Hotspot' data."""
    print("🔬 TEST MODE: Generating 'Pulsing Hotspot' data...")
    df, timestamps, sensor_cols, positions, control_col, test_name = _create_base_test_df(config, "Pulsing Hotspot Test")
    
    base_temp, max_intensity = 800.0, 40.0
    hotspot_pos = np.array([0.7, 0.3])
    
    # Vectorized calculation
    time_progress = np.linspace(0, 1, len(timestamps))
    pulse = (np.sin(time_progress * 20 * (1 + time_progress * 3)) + 1) / 2
    current_intensity = max_intensity * pulse
    
    distances_sq = np.sum((positions - hotspot_pos)**2, axis=1)
    spatial_falloff = np.exp(-distances_sq / 0.05)
    
    # Broadcasting intensity over spatial falloff
    effect = current_intensity[:, np.newaxis] * spatial_falloff
    
    temp_data = base_temp + effect
    df[sensor_cols] = temp_data

    for col in config["requested_columns"]:
        if col not in df.columns:
            df[col] = base_temp
    df[control_col] = base_temp
            
    df['source_file'] = test_name
    print("✅ Test data generated successfully.")
    return df

print("✅ Advanced and vectorized test data generator suite defined.")

In [ ]:
class ThermocoupleAnalyzer:
    """
    An object-oriented class to load, analyze, and visualize thermocouple data
    with interactive plots and GIF export capabilities.
    """
    def __init__(self, file_paths, config, imageio_available=False):
        if file_paths and not isinstance(file_paths, (list, tuple)):
            file_paths = [file_paths]
        self.file_paths = file_paths
        self.config = config
        self.imageio_available = imageio_available
        
        # Centralize key configuration values
        self.control_col = self.config.get("control_thermocouple", "70418_T901000_W1T")
        self.exclude_from_drift = self.config.get("exclude_from_drift", ["70418_T901100_X1"])
        
        self.data = None
        self.available_columns, self.missing_columns, self.color_map = [], [], {}
        self.colors = self.config.get("custom_colors") or px.colors.qualitative.D3
        self.figures = {}

    def load_data(self):
        """
        Loads data from one or more Excel or CSV files, using a robust manual header detection method.
        """
        all_dataframes, all_found_columns = [], set()
        for file_path in self.file_paths:
            print(f"--- Processing file: {os.path.basename(file_path)} ---")
            try:
                # Determine file type and read a small portion to find the header
                is_excel = file_path.lower().endswith((".xlsx", ".xls"))
                df_peek = pd.read_excel(file_path, header=None, nrows=60, dtype=str) if is_excel else pd.read_csv(file_path, sep=';', header=None, nrows=60, dtype=str, engine='python')
                
                # Manually find the header row by looking for 'TimeStamp'
                header_idx = next((i for i, r in enumerate(df_peek.itertuples(index=False)) if any("TimeStamp" in str(v) for v in r)), None)
                
                if header_idx is None:
                    print("  -> 'TimeStamp' header not found in the first 60 rows. Skipping.")
                    continue
                
                # Get the header content and determine how many rows to skip in the full read
                skip_rows = header_idx + 1
                header_row = df_peek.iloc[header_idx].fillna("").astype(str).str.strip().tolist()

                # Read the actual data, skipping the header and rows above it
                df = pd.read_excel(file_path, header=None, skiprows=skip_rows, dtype=str) if is_excel else pd.read_csv(file_path, sep=';', header=None, skiprows=skip_rows, dtype=str, engine='python')
                
                # Clean up and assign the manually found header
                while header_row and (not header_row[-1] or pd.isna(header_row[-1])):
                    header_row.pop()
                df = df.iloc[:, :len(header_row)]
                df.columns = header_row
                df = df.dropna(axis=0, how="all").reset_index(drop=True)

                if df.empty:
                    print("  -> File is empty after loading. Skipping.")
                    continue

                # Identify timestamp and time columns from the now-assigned header
                ts_col = header_row[0]
                t_col = (header_row[1] if len(header_row) > 1 and df[header_row[1]].astype(str).str.contains(":").any() else None)

                # Parse the datetime column
                if t_col:
                    df["datetime"] = _parse_timestamp(df[ts_col].astype(str).str.strip() + " " + df[t_col].astype(str).str.strip())
                else:
                    df["datetime"] = _parse_timestamp(df[ts_col].astype(str).str.strip())
                
                df = df.dropna(subset=["datetime"])

                # Convert all other relevant columns to numeric
                for col in [c for c in df.columns if c not in ["datetime", ts_col, t_col]]:
                    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ".", regex=False).str.strip(), errors="coerce")
                
                df['source_file'] = os.path.basename(file_path)
                all_dataframes.append(df)
                all_found_columns.update(df.columns)

            except FileNotFoundError:
                print(f"  -> ERROR: File not found at {file_path}. Skipping.")
            except Exception as e:
                print(f"  -> ERROR: Could not read file. Skipping. Details: {e}")

        if not all_dataframes:
            raise ValueError("Could not load any valid data from the selected files.")

        self.data = pd.concat(all_dataframes, ignore_index=True).sort_values('datetime').reset_index(drop=True)
        
        req_cols = self.config.get("requested_columns", [])
        self.available_columns = [c for c in req_cols if c in all_found_columns]
        self.missing_columns = [c for c in req_cols if c not in all_found_columns]

        print(f"\n✅ Loading successful. Combined {len(self.data)} rows from {len(self.file_paths)} file(s).")
        if self.missing_columns:
            print(f"⚠️ Warning: The following requested columns were not found: {self.missing_columns}")

    def show_summary(self):
        if self.data is None: return
        for source_file in self.data['source_file'].unique():
            print(f"\n--- Average Temperature Summary for: {source_file} ---")
            summary_df = self.data[self.data['source_file'] == source_file][self.available_columns].mean().round(2)
            print(pd.DataFrame(summary_df, columns=['Average Temp (°C)']))
            
    def perform_advanced_analysis(self):
        if self.data is None: return
        if self.control_col not in self.data.columns:
            print(f"\n⚠️ Advanced Analysis Skipped: Control column '{self.control_col}' not found in data.")
            return

        print("\n🔬 Performing Advanced Analysis (calculating drift)...")
        self.drift_columns = []
        for col in self.available_columns:
            if col != self.control_col and col not in self.exclude_from_drift:
                drift_col_name = f"Drift_{col}"
                self.drift_columns.append(drift_col_name)
                self.data[drift_col_name] = self.data[col] - self.data[self.control_col]
        
        if self.drift_columns:
            print("\n--- Advanced Analysis Summary ---")
            summary_df = self.data[self.drift_columns].mean().round(2)
            print(pd.DataFrame(summary_df, columns=['Average Drift from Control (°C)']))

    def _get_base_layout_settings(self):
        """Returns a dictionary with base layout settings for plots."""
        return dict(
            height=700,
            plot_bgcolor='#f0f0f0',
            paper_bgcolor='white',
            font=dict(family="Arial, sans-serif", size=14, color="black"),
            xaxis=dict(title_text="Time", title_font_size=18, showline=True, linecolor='black', mirror=True, showgrid=True, gridwidth=1, gridcolor='white'),
            yaxis=dict(title_text="Temperature (°C)", title_font_size=18, showline=True, linecolor='black', mirror=True, showgrid=True, gridwidth=1, gridcolor='white'),
            legend=dict(title_text='<b>Thermocouples</b>', title_font_size=16, bordercolor="Black", borderwidth=1, bgcolor='rgba(255,255,255,0.8)'),
            hovermode="x unified",
            hoverlabel=dict(bgcolor="white", bordercolor="black", font_size=14)
        )

    def plot_raw_temperatures(self):
        if self.data is None or not self.available_columns: return
        print("\nGenerating main temperature plot...")
        fig = go.Figure()
        
        layout_settings = self._get_base_layout_settings()
        layout_settings['title'] = dict(text=self.config.get('plot_title', "<b>Temperature vs. Time</b>"), x=0.5, xanchor='center', font=dict(size=24, family="Arial Black, sans-serif"))
        fig.update_layout(**layout_settings)

        sorted_cols = self.data[self.available_columns].mean().sort_values(ascending=False).index.tolist()
        legend_map = self.config.get("legend_map", {})
        dashed_col = self.config.get("dashed_column")
        self.color_map = {}

        for idx, col in enumerate(sorted_cols):
            color = self.colors[idx % len(self.colors)]
            self.color_map[col] = color
            fig.add_trace(go.Scatter(
                x=self.data["datetime"], 
                y=self.data[col], 
                name=legend_map.get(col, col), 
                mode='lines', 
                line=dict(width=2.5, dash='dash' if col == dashed_col else 'solid', color=color), 
                hovertemplate='<b>%{y:.2f}°C</b><br>%{x|%d-%b-%Y %H:%M:%S}<extra></extra>'
            ))
        self.figures['Raw Temperatures vs. Time'] = fig
        fig.show(config=self.config.get("save_config"))

    def plot_drift(self):
        if not hasattr(self, 'drift_columns') or not self.drift_columns: return
        print("\nGenerating drift plot...")
        fig = go.Figure()
        
        layout_settings = self._get_base_layout_settings()
        layout_settings['title'] = dict(text="<b>Thermocouple Drift vs. Time</b>", x=0.5, xanchor='center', font=dict(size=24, family="Arial Black, sans-serif"))
        layout_settings['yaxis']['title_text'] = "Drift from Control (°C)"
        fig.update_layout(**layout_settings)

        legend_map = self.config.get("legend_map", {})
        sorted_cols = self.data[self.drift_columns].abs().mean().sort_values(ascending=False).index.tolist()

        for col in sorted_cols:
            orig_col = col.replace("Drift_", "")
            line_color = self.color_map.get(orig_col, 'grey')
            fig.add_trace(go.Scatter(
                x=self.data["datetime"], 
                y=self.data[col], 
                name=legend_map.get(orig_col, orig_col), 
                mode='lines', 
                line=dict(width=2.5, color=line_color), 
                hovertemplate='<b>Drift: %{y:+.2f}°C</b><br>%{x|%d-%b-%Y %H:%M:%S}<extra></extra>'
            ))
        self.figures['Temperature Drift vs. Time'] = fig
        fig.show(config=self.config.get("save_config"))
    
    def _prepare_contour_traces(self, mean_temps, is_reference=False):
        spatial_map = self.config.get("spatial_map")
        if not spatial_map: return None, None, 0

        # Use the class attribute for the control column
        avg_control_temp = 0 if is_reference else mean_temps.get(self.control_col, 0)
        
        points, values, hover_texts = [], [], []
        for col, temp in mean_temps.items():
            if col in spatial_map:
                deviation = temp - avg_control_temp
                points.append(spatial_map[col])
                values.append(deviation)
                hover_texts.append(f"<b>{self.config['legend_map'].get(col, col)}</b><br>Avg Dev: {deviation:+.2f}°C")

        if len(points) < 3: return None, None, 0

        points, values = np.array(points), np.array(values)
        grid_x, grid_y = np.mgrid[min(points[:,0]):max(points[:,0]):100j, min(points[:,1]):max(points[:,1]):100j]
        
        try:
            grid_z = griddata(points, values, (grid_x, grid_y), method='cubic')
        except Exception as e:
            print(f"  -> Warning: Could not perform 'cubic' interpolation, falling back to 'linear'. Reason: {e}")
            grid_z = griddata(points, values, (grid_x, grid_y), method='linear')


        contour_trace = go.Contour(z=grid_z.T, x=grid_x[:,0], y=grid_y[0,:], colorscale='RdBu_r', colorbar=dict(title='Deviation (°C)'), zmid=0)
        scatter_trace = go.Scatter(x=points[:,0], y=points[:,1], mode='markers', marker=dict(size=12, color='rgba(0,0,0,0.6)', line=dict(width=1, color='White')), text=hover_texts, hoverinfo='text', showlegend=False)
        
        return contour_trace, scatter_trace, np.nanmax(np.abs(values))

    def plot_contour(self):
        if self.data is None: return
        print("\nGenerating average deviation contour plot...")
        
        contour_trace, scatter_trace, max_abs_val = self._prepare_contour_traces(self.data[self.available_columns].mean())
        if not contour_trace:
            print("⚠️ Contour Plot Skipped: Not enough spatial data points (at least 3 required).")
            return
            
        fig = go.Figure(data=[contour_trace, scatter_trace])
        fig.update_traces(selector=dict(type='contour'), zmin=-max_abs_val, zmax=max_abs_val)
        
        # --- PLOT FIX ---
        # Set fixed, padded ranges and maintain a square aspect ratio.
        fig.update_layout(
            title="<b>Average Temperature Deviation from Control</b>",
            height=800,
            template='plotly_white',
            showlegend=False,
            xaxis=dict(range=[-0.2, 1.2], scaleanchor="y", scaleratio=1),
            yaxis=dict(range=[-0.2, 1.2])
        )
        self.figures['Average Temperature Deviation'] = fig
        fig.show(config=self.config.get("save_config"))

    def plot_side_by_side_comparison(self, reference_data_dict):
        print("\nGenerating side-by-side contour comparison...")
        fig = make_subplots(rows=1, cols=2, subplot_titles=("<b>Current Run</b>", "<b>Reference Run</b>"))
        
        trace1, scatter1, max_dev1 = self._prepare_contour_traces(self.data[self.available_columns].mean())
        if trace1:
            fig.add_trace(trace1, row=1, col=1)
            fig.add_trace(scatter1, row=1, col=1)

        trace2, scatter2, max_dev2 = self._prepare_contour_traces(pd.Series(reference_data_dict), is_reference=True)
        if trace2:
            fig.add_trace(trace2, row=1, col=2)
            fig.add_trace(scatter2, row=1, col=2)

        if not trace1 and not trace2:
            print("⚠️ Comparison Plot Skipped: No valid data for either plot.")
            return

        abs_max = max(max_dev1, max_dev2, 1) # Use 1 as a fallback
        fig.update_traces(selector=dict(type='contour'), zmin=-abs_max, zmax=abs_max)

        # --- PLOT FIX ---
        # Apply fixed ranges and square aspect ratio to BOTH subplots.
        fig.update_xaxes(range=[-0.2, 1.2])
        fig.update_yaxes(range=[-0.2, 1.2], scaleanchor="x", scaleratio=1)
        
        fig.update_layout(title_text="<b>Side-by-Side Comparison to Reference</b>", height=800, template='plotly_white')
        self.figures['Side-by-Side Comparison'] = fig
        fig.show(config=self.config.get("save_config"))

    def _prepare_dynamic_contour_data(self, num_frames=75):
        """Helper function to prepare data for dynamic plots, avoiding code duplication."""
        spatial_map = self.config.get("spatial_map")
        if self.data is None or not spatial_map or self.control_col not in self.data.columns:
            print("\n⚠️ Dynamic Contour Plot Skipped: Missing spatial map, data, or control column.")
            return None

        contour_cols = [c for c in self.available_columns if c in spatial_map]
        if len(contour_cols) < 3:
            print("\n⚠️ Dynamic Contour Plot Skipped: Fewer than 3 mappable sensors found.")
            return None
        
        points = np.array([spatial_map[col] for col in contour_cols])
        all_deviations = self.data[contour_cols].sub(self.data[self.control_col], axis=0)
        
        max_abs_deviation = all_deviations.abs().max().max()
        if pd.isna(max_abs_deviation) or max_abs_deviation == 0:
            max_abs_deviation = 1.0

        unique_timestamps = self.data['datetime'].unique()
        step = max(1, len(unique_timestamps) // num_frames)
        animation_timestamps = unique_timestamps[::step]
        
        grid_x, grid_y = np.mgrid[min(points[:,0]):max(points[:,0]):100j, min(points[:,1]):max(points[:,1]):100j]
        
        all_z_data = []
        for ts in animation_timestamps:
            data_at_ts = self.data.loc[self.data['datetime'] == ts]
            devs = data_at_ts[contour_cols].iloc[0].values - data_at_ts[self.control_col].iloc[0]
            try:
                grid_z = griddata(points, devs, (grid_x, grid_y), method='cubic').T
            except:
                grid_z = griddata(points, devs, (grid_x, grid_y), method='linear').T
            all_z_data.append(grid_z)
            
        return {
            "grid_x": grid_x, "grid_y": grid_y, "all_z_data": all_z_data,
            "animation_timestamps": animation_timestamps, "max_abs_deviation": max_abs_deviation,
            "points": points, "contour_cols": contour_cols
        }

    def plot_dynamic_contour(self, num_frames=75):
        print(f"\nGenerating dynamic contour plot with up to {num_frames} frames...")
        
        plot_data = self._prepare_dynamic_contour_data(num_frames)
        if not plot_data: return

        print(f"Animation color scale locked to: ±{plot_data['max_abs_deviation']:.2f}°C")

        # Unpack data for clarity
        grid_x, grid_y, all_z_data = plot_data['grid_x'], plot_data['grid_y'], plot_data['all_z_data']
        timestamps, max_dev = plot_data['animation_timestamps'], plot_data['max_abs_deviation']

        fig = go.Figure(data=[go.Contour(
            z=all_z_data[0], x=grid_x[:,0], y=grid_y[0,:], colorscale='RdBu_r',
            colorbar=dict(title='Deviation (°C)'), zmid=0, zmin=-max_dev, zmax=max_dev
        )])
        
        frames = [go.Frame(data=[go.Contour(z=z_data)], name=str(ts)) for z_data, ts in zip(all_z_data, timestamps)]
        slider_steps = [{"method": "animate", "args": [[str(ts)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}], "label": pd.to_datetime(ts).strftime('%H:%M:%S')} for ts in timestamps]

        fig.update(frames=frames)
        fig.update_layout(
            title="<b>Dynamic Temperature Deviation from Control</b>",
            # --- PLOT FIX --- (Applying consistent formatting here too)
            xaxis=dict(range=[-0.2, 1.2], scaleanchor="y", scaleratio=1),
            yaxis=dict(range=[-0.2, 1.2]),
            height=800, template='plotly_white',
            updatemenus=[dict(
                type="buttons", showactive=False, x=0.1, y=-0.05, xanchor="right", yanchor="top",
                buttons=[
                    dict(label="Play", method="animate", args=[None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}])
                ])],
            sliders=[dict(
                active=0, steps=slider_steps, x=0.1, len=0.9,
                xanchor="left", y=-0.1, yanchor="top"
            )]
        )
        fig.show(config=self.config.get("save_config"))

    def export_dynamic_contour_as_gif(self, num_frames=75, export_folder="."):
        if not self.imageio_available:
            print("❌ GIF Export Failed: 'imageio' library is not installed.")
            return
        print(f"\n🎬 Preparing to export animation as GIF...")
        plot_data = self._prepare_dynamic_contour_data(num_frames)
        if not plot_data: return
        if not export_folder or not os.path.isdir(export_folder):
            export_folder = "."
            print(f"⚠️ Warning: Invalid export folder. Defaulting to current directory: {os.path.abspath(export_folder)}")
        filename = os.path.join(export_folder, "dynamic_contour_animation.gif")
        images = []
        max_dev = plot_data['max_abs_deviation']
        
        # --- DYNAMIC FRAME COUNTER ---
        total_frames = len(plot_data['animation_timestamps'])
        print(f"Stitching {total_frames} frames into a GIF (in-memory)...")

        for i, ts in enumerate(plot_data['animation_timestamps']):
            # Print the progress, updating the line with '\r'
            print(f"  -> Processing frame {i + 1} of {total_frames}...", end='\r', flush=True)
            
            fig = go.Figure(data=[go.Contour(z=plot_data['all_z_data'][i], x=plot_data['grid_x'][:,0], y=plot_data['grid_y'][0,:], colorscale='RdBu_r', colorbar=dict(title='Deviation (°C)'), zmid=0, zmin=-max_dev, zmax=max_dev)])
            fig.update_layout(
                title=f"<b>Temp Dev ({pd.to_datetime(ts).strftime('%H:%M:%S')})</b>", template='plotly_white',
                xaxis=dict(range=[-0.2, 1.2], scaleanchor="y", scaleratio=1), yaxis=dict(range=[-0.2, 1.2])
            )
            image_bytes = fig.to_image(format='png', scale=2)
            images.append(imageio.imread(image_bytes))

        # After the loop, print a newline to move on from the progress bar
        print() 
        imageio.mimsave(filename, images, duration=150, loop=0)
        print(f"✅ Successfully saved animation to '{os.path.abspath(filename)}'!")
    
    def export_plots_to_pptx(self, export_folder="."):
        """
        Exports plots to a PowerPoint presentation with images hyperlinked to
        interactive HTML files.
        """
        try:
            from pptx import Presentation
            from pptx.util import Inches, Pt
            from io import BytesIO
        except ImportError:
            print("❌ PowerPoint Export Failed: 'python-pptx' library is not installed.")
            print("   To enable this feature, run '!pip install python-pptx' in a code cell and restart.")
            return

        if not export_folder or not os.path.isdir(export_folder):
            export_folder = "."
        
        # Define paths for the presentation and a subfolder for the HTML files
        pptx_filename = os.path.join(export_folder, "Interactive_Analysis_Report.pptx")
        html_export_path = os.path.join(export_folder, "interactive_html_plots")
        os.makedirs(html_export_path, exist_ok=True)
        
        print(f"\n📎 Creating interactive PowerPoint presentation...")
        print(f"   -> Saving presentation to: '{pptx_filename}'")
        print(f"   -> Saving HTML files to: '{html_export_path}'")

        prs = Presentation()
        prs.slide_width = Inches(16)
        prs.slide_height = Inches(9)
        title_slide_layout = prs.slide_layouts[0]
        blank_layout = prs.slide_layouts[6]

        # --- Title Slide ---
        slide = prs.slides.add_slide(title_slide_layout)
        title = slide.shapes.title
        subtitle = slide.placeholders[1]
        title.text = "Interactive Thermocouple Analysis Report"
        first_file = os.path.basename(self.file_paths[0]) if self.file_paths else "Test Data"
        subtitle.text = f"Click on plot images to open interactive versions.\nAnalysis of: {first_file}\nGenerated on: {pd.Timestamp.now().strftime('%Y-%m-%d')}"

        # --- Add a slide for each figure ---
        for fig_name, fig in self.figures.items():
            slide = prs.slides.add_slide(blank_layout)
            
            # Add title
            title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(15), Inches(0.8))
            p = title_shape.text_frame.paragraphs[0]
            p.text = fig_name
            p.font.bold = True
            p.font.size = Pt(24)

            # --- Save both image and interactive HTML ---
            image_stream = BytesIO()
            fig.write_image(image_stream, format='png', scale=2)
            image_stream.seek(0)
            
            html_filename = f"{fig_name.replace(' ', '_').lower()}.html"
            html_filepath = os.path.join(html_export_path, html_filename)
            fig.write_html(html_filepath)

            # --- Add hyperlinked picture ---
            pic = slide.shapes.add_picture(image_stream, Inches(0.5), Inches(1.25), height=Inches(6.5))
            pic.left = int((prs.slide_width - pic.width) / 2) # Center horizontally
            
            # Add the hyperlink action to the picture
            pic.click_action.hyperlink.address = os.path.abspath(html_filepath)
            
            image_stream.close()

        prs.save(pptx_filename)
        print(f"✅ Successfully saved presentation!")

    def run(self, reference_data=None):
        try:
            # Step 1: Load data if it hasn't been loaded already
            if self.data is None:
                self.load_data()

            # Step 2: Run all the standard analysis and plotting functions
            self.show_summary()
            self.perform_advanced_analysis()
            self.plot_raw_temperatures()
            self.plot_drift()
            self.plot_contour()

            # Step 3: Run the comparison plot if reference data is provided
            if reference_data is not None:
                self.plot_side_by_side_comparison(reference_data)
            else:
                print("\nℹ️ Side-by-side comparison skipped (no reference data provided).")

            # Step 4: Interactively ask the user about creating a dynamic plot
            print("\n--- Optional: Dynamic Contour Plot ---")
            quality_map = {'1': 500, '2': 250, '3': 120, '4': 60, 'n': 0}
            quality_desc = {'1': "High (500 frames)", '2': "Good (250 frames)", '3': "Medium (120 frames)", '4': "Low (60 frames)"}
            chosen_frames = 0
            while True:
                print("Select animation quality:")
                for k, v in quality_desc.items():
                    print(f"  [{k}] {v}")
                print("  [n] No, skip animation.")
                choice = input("Enter your choice [1-4, n]: ").lower().strip()

                if choice in quality_map:
                    chosen_frames = quality_map[choice]
                    if chosen_frames > 0:
                        self.plot_dynamic_contour(num_frames=chosen_frames)
                    else:
                        print("Dynamic contour plot skipped.")
                    break
                else:
                    print("Invalid input. Please try again.")

            # Step 5: If a dynamic plot was created, ask about exporting it as a GIF
            if chosen_frames > 0 and self.imageio_available:
                if input("\nWould you like to save this animation as a GIF? (This can be slow) [y/n]: ").lower().strip() in ['y', 'yes']:
                    export_folder = self.config.get('export_folder_path')
                    self.export_dynamic_contour_as_gif(num_frames=chosen_frames, export_folder=export_folder)

            # Step 6: --- NEW: PowerPoint Export Option ---
            if input("\nWould you like to export the plots to an interactive PowerPoint presentation? [y/n]: ").lower().strip() in ['y', 'yes']:
                export_folder = self.config.get('export_folder_path')
                self.export_plots_to_pptx(export_folder=export_folder)

        except Exception as e:
            print(f"\n--- An error occurred during execution ---\nError Type: {type(e).__name__}\nError Details: {e}")

print("✅ ThermocoupleAnalyzer class defined.")

In [ ]:
REFERENCE_DEVIATION_DATA = {
    "70418_T654001_X1": -8.82,  # Kathodenblock
    "70418_T654002_X1": 5.08,   # Cell Holder, Top-Left
    "70418_T654003_X1": 19.28,  # Cell Holder, Bottom-Left
    "70418_T654004_X1": -4.42,   # Cell Holder, Middle-Right
    "70418_T654008_X1": 2.38,   # Cell Holder, Top-Right
    "70418_T654009_X1": -13.42,   # Cell Holder, Bottom-Right
    "70418_T901000_X1": 0.06,  # Zell Temperatur
}
print("✅ Reference data for comparison loaded.")

In [ ]:
USER_CONFIG = {
    "requested_columns": [
        "70418_T654001_X1", "70418_T654002_X1", "70418_T654003_X1", "70418_T654004_X1", 
        "70418_T654008_X1", "70418_T654009_X1", "70418_T901000_W1T", "70418_T901000_X1", 
        "70418_T901100_X1"
    ],
    "legend_map": {
        "70418_T654001_X1": "Kathodenblock", "70418_T654002_X1": "Cell Holder, Top-Left",
        "70418_T654003_X1": "Cell Holder, Bottom-Left", "70418_T654004_X1": "Cell Holder, Middle-Right",
        "70418_T654008_X1": "Cell Holder, Top-Right", "70418_T654009_X1": "Cell Holder, Bottom-Right",
        "70418_T901000_W1T": "Control Thermocouple", "70418_T901000_X1": "Zell Temperatur",
        "70418_T901100_X1": "Ofen Temperatur"
    },
    "spatial_map": {
        "70418_T654001_X1": (0.5, 0.5), "70418_T654002_X1": (0, 1), "70418_T654003_X1": (0, 0),
        "70418_T654004_X1": (1, 0.5), "70418_T654008_X1": (1, 1), "70418_T654009_X1": (1, 0),
        "70418_T901000_X1": (0, 0.5)
    },
    "control_thermocouple": "70418_T901000_W1T",
    "exclude_from_drift": ["70418_T901100_X1"],
    "dashed_column": "70418_T901000_W1T",
    "plot_title": "<b>Temperature vs. Time</b>",
    "save_config": {'toImageButtonOptions': {'format': 'svg', 'filename': 'custom_plot_export', 'scale': 3}}
}
print("✅ User configuration loaded.")

In [ ]:
def main():
    # SET TO 'True' to run the test, 'False' for normal operation with your files.
    RUN_TEST_MODE = True

    try:
        ref_data = REFERENCE_DEVIATION_DATA
    except NameError:
        ref_data = None

    # --- Graphical Folder/File Picker Logic ---
    export_path = None
    if input("Do you want to select a custom folder to save GIFs? [y/n]: ").lower().strip() in ['y', 'yes']:
        print("Opening folder selection dialog...")
        
        try:
            from ctypes import windll
            windll.shcore.SetProcessDpiAwareness(1)
        except Exception:
            pass # Fails gracefully on non-Windows OS

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        export_path = filedialog.askdirectory(parent=root, title="Select a folder for your exports")
        root.destroy()
        
        if export_path:
            print(f"✅ Exports will be saved to: {export_path}")
        else:
            print("⚠️ No folder selected. Exports will save in the current directory.")

    USER_CONFIG['export_folder_path'] = export_path

    if RUN_TEST_MODE:
        while True:
            print("\n--- Select a Test Scenario ---")
            print("  [1] Fast Center-Out Ripple")
            print("  [2] Interfering Waves (Complex)")
            print("  [3] Pulsing Hotspot (Dramatic)")
            choice = input("Enter your choice [1-3]: ").strip()
            
            if choice == '1': test_df = generate_center_out_ripple_data(USER_CONFIG); break
            elif choice == '2': test_df = generate_interfering_waves_data(USER_CONFIG); break
            elif choice == '3': test_df = generate_pulsing_hotspot_data(USER_CONFIG); break
            else: print("Invalid input.")

        # Initialize analyzer in test mode
        analyzer = ThermocoupleAnalyzer(file_paths=["Test Data"], config=USER_CONFIG, imageio_available=IMAGEIO_AVAILABLE)
        analyzer.data = test_df # Manually inject the generated data
        analyzer.available_columns = [c for c in USER_CONFIG['requested_columns'] if c in test_df.columns]
        analyzer.run(reference_data=ref_data)
    else:
        # High-DPI awareness for better dialog appearance on Windows
        try:
            from ctypes import windll
            windll.shcore.SetProcessDpiAwareness(1)
        except Exception:
            pass # Fails gracefully on non-Windows OS

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        print("Opening file selection dialog...")
        file_paths = filedialog.askopenfilenames(
            parent=root,
            title="Select One or More Thermocouple Data Files",
            filetypes=(("Excel files", "*.xlsx *.xls"), ("CSV files", "*.csv"), ("All files", "*.*"))
        )
        root.destroy()
        
        if file_paths:
            print(f"{len(file_paths)} file(s) selected.")
            analyzer = ThermocoupleAnalyzer(file_paths, USER_CONFIG, imageio_available=IMAGEIO_AVAILABLE)
            analyzer.run(reference_data=ref_data)
        else:
            print("No files were selected. Exiting program.")

# This block allows the script to be run from a terminal or notebook
if __name__ == '__main__':
    main()